# Data pre-processing

**Dataset:** Sanger2024

### Libraries & directories

In [1]:
# libraries
import pandas
import sys
import os

# Set working directory to the main folder of the project
main_folder = "C:/Users/viviamsb/OneDrive - NTNU/PhD Folder/Pipeline/trafikk_paper"
os.chdir(main_folder)

# Print the current working directory
print("\nCurrent Working Directory:", os.getcwd())


Current Working Directory: C:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\trafikk_paper


In [2]:
# Directories

    # input
synergy_metadata_file = "data/metadata/Sanger-2024/raw/vis2024_processed.tsv"
drugscreen_metadata_file = "data/metadata/Sanger-2024/raw/S1_cell_line_drug_info.xlsx"

    # output
metadata_output_dir = "data/metadata/Sanger-2024/"

### Cell lines and tissues

In [3]:
#//////////////////////////////////////// PROCESS CELL LINES //////////////////////////////////////////////////

cell_line_df = pandas.read_excel(drugscreen_metadata_file, sheet_name='TabS1A')
    # Make the list
cell_line_df = cell_line_df[['model_id', 'model_name', 'tissue']].drop_duplicates().reset_index(drop=True)
cell_line_df = cell_line_df.rename(columns={'model_id': 'SIDM', 'model_name': 'cell_line_name', 'tissue': 'Tissue'})

    # Change tissue names to short versions
tissue_mapping = {
    'Central Nervous System': 'CNS',
    'Peripheral Nervous System': 'PNS',
    'Haematopoietic and Lymphoid': 'Lymphoid',
    'Head and Neck': 'Head',
    'Large Intestine': 'Colon',
    'Soft Tissue': 'Soft_tissue',
    'Biliary Tract': 'Biliary',
    'Adrenal Gland': 'Adrenal',
}
cell_line_df['Tissue'] = cell_line_df['Tissue'].map(tissue_mapping).fillna(cell_line_df['Tissue'])


    # Save the cell line list
cell_line_df.to_csv(metadata_output_dir + 'clines_tissue.csv', index=False)
cell_line_list = cell_line_df['cell_line_name'].tolist()


print('\nCELL LINES')
print('Number of cell lines:', len(cell_line_list))
print('Cell line names:', cell_line_list)



CELL LINES
Number of cell lines: 757
Cell line names: [697, 5637, '22RV1', '23132-87', '42-MG-BA', '639-V', '647-V', '769-P', '786-0', '8305C', '8505C', '8-MG-BA', 'A101D', 'A172', 'A204', 'A2058', 'A253', 'A2780', 'A375', 'A3-KAW', 'A498', 'A4-Fuk', 'A549', 'A673', 'A704', 'ACHN', 'AGS', 'ALL-PO', 'ALL-SIL', 'AM-38', 'AMO-1', 'AN3-CA', 'ARH-77', 'ASH-3', 'AsPC-1', 'ATN-1', 'AU565', 'BALL-1', 'BB49-HNC', 'BC-1', 'B-CPAP', 'BE-13', 'BE2-M17', 'Becker', 'BFTC-905', 'BFTC-909', 'BHT-101', 'BICR10', 'BICR22', 'BICR78', 'BL-41', 'BPH-1', 'BT-20', 'BT-474', 'BT-483', 'BT-549', 'BV-173', 'BxPC-3', 'C2BBe1', 'C32', 'C-33-A', 'C3A', 'C-4-I', 'CA46', 'Ca9-22', 'CADO-ES1', 'CAL-120', 'CAL-148', 'CAL-27', 'CAL-29', 'CAL-33', 'CAL-39', 'CAL-51', 'CAL-54', 'CAL-62', 'CAL-72', 'CAL-78', 'CAL-85-1', 'Calu-3', 'Calu-6', 'CAMA-1', 'Caov-4', 'CAPAN-1', 'CAPAN-2', 'CaR-1', 'CAS-1', 'Ca-Ski', 'CCK-81', 'CCRF-CEM', 'CESS', 'CFPAC-1', 'CGTH-W-1', 'CHL-1', 'CHP-134', 'CHP-212', 'CHSA8926', 'CL-11', 'CL-40', 

### Drug information

In [4]:
#//////////////////////////////////////// PROCESS DRUGS //////////////////////////////////////////////////

# Get CHEMBL_IDs
drug_info_df = pandas.read_excel(drugscreen_metadata_file, sheet_name='TabS1C')

druginfo = drug_info_df[['Name', 'Chembl', 'Target name', 'Target']].reset_index(drop=True)
druginfo = druginfo.rename(columns={'Name': 'drug_name', 'Chembl': 'ChEMBL_ID', 'Target name': 'target_name', 'Target': 'target'})

# Clean the format of the target column
druginfo['target'] = druginfo['target'].str.replace(';', ', ')
# Join the target columns
druginfo['target'] = druginfo['target_name'] + ', ' + druginfo['target']
druginfo.drop(columns=['target_name'], inplace=True)
druginfo.to_csv(metadata_output_dir + 'drug_info.csv', index=False)

### Synergy data

In [5]:
#//////////////////////////////////////// PROCESS SYNERGY SCORES //////////////////////////////////////////////////

synergy_df = pandas.read_csv(synergy_metadata_file, sep='\t')
print(f'\nSYNERGY DATA:\n{synergy_df.columns}')

synergies = synergy_df[['tissue', 'CELL_LINE_NAME', 'ANCHOR_NAME', 'LIBRARY_NAME', 'BLISS']].copy()
rename_dict = {
    'CELL_LINE_NAME': 'cell_line',
    'ANCHOR_NAME': 'drug_name_A',
    'LIBRARY_NAME': 'drug_name_B',
    'BLISS': 'synergy',
}
synergies = synergies.rename(columns=rename_dict)

    # In this dataset, some tissues are grouped together under "Other".
    # We want to keep the respective tissue name, so we map "Other" to the correct tissue names based on cell line names.
    # We will reuse the cell_line_df since it has tissue and cell line name information.
synergies = synergies.merge(cell_line_df[['cell_line_name', 'Tissue']], left_on='cell_line', right_on='cell_line_name', how='left')
synergies['tissue'] = synergies.apply(lambda row: row['Tissue'] if row['tissue'] == 'Other' else row['tissue'], axis=1)
synergies.drop(columns=['cell_line_name', 'Tissue'], inplace=True)


# Change tissue names to short versions
synergies['tissue'] = synergies['tissue'].map(tissue_mapping).fillna(synergies['tissue'])

# Save the synergy data
synergies.to_csv(metadata_output_dir + 'synergy_data.csv', index=False, header=True)

print('\nSYNERGY DATA PROCESSED:\n')
print(f'Columns: {synergies.columns}')
print(synergies.head())


SYNERGY DATA:
Index(['CELL_LINE_NAME', 'tissue', 'ANCHOR_ID', 'LIBRARY_ID',
       'ANCHOR_VIABILITY', 'LIBRARY_EMAX', 'SYNERGY_OBS_EMAX',
       'SYNERGY_DELTA_EMAX', 'SYNERGY_DELTA_XMID', 'SYNERGY_DELTA_IC50',
       'SYNERGY_EMAX', 'BLISS_VIS', 'HSA_VIS', 'HSA', 'BLISS', 'SUB_HSA',
       'NO_SYNERGY_VIS', 'NO_BLISS_OR_HSA', 'ANCHOR_NAME', 'LIBRARY_NAME',
       'ANCHOR_GENE_TARGET', 'LIBRARY_GENE_TARGET'],
      dtype='str')

SYNERGY DATA PROCESSED:

Columns: Index(['tissue', 'cell_line', 'drug_name_A', 'drug_name_B', 'synergy'], dtype='str')
    tissue cell_line drug_name_A drug_name_B  synergy
0  Bladder      5637     AZD7762   Cisplatin        1
1  Bladder     639-V     AZD7762   Cisplatin        1
2  Bladder     647-V     AZD7762   Cisplatin        1
3  Bladder  BFTC-905     AZD7762   Cisplatin        1
4  Bladder    CAL-29     AZD7762   Cisplatin        1
